In [1]:
from slrrag.querier_openalex import query_openalex
import requests
import pandas as pd
import plotly.express as px
import os

%load_ext dotenv
%dotenv
input_query_openalex = """((("offshore") AND ("infrastructure" OR "construction" OR "installation" OR "windfarm" OR "wind farm" OR "windmill" OR "wind turbine" OR "wind park" OR "wind energy" OR "platform" OR "quay" OR "wharf" OR "lidar")) OR ("bridge" OR "flotel" OR "Aquaculture farm" OR "Artificial Island" OR "Synthetic Island")) AND
(("risk" OR "safe" OR "safety") AND ("assess" OR "assessing" OR "assessment" OR "model" OR "modeling" OR "modelisation" OR "predict" OR "predicting" OR "prediction" OR "analyse" OR "analysing" OR "analyze" OR "analyzing" OR "study" OR "studies" OR "studying") AND ("ship" OR "vessel" OR "maritime traffic" OR "navigation" OR "navigational" OR "nautical"))
"""
l_test_openalex = query_openalex(input_query_openalex, ask_input=False, n_paper_max=1)
l_test_openalex[0]

cannot find .env file
7638 found in OpenAlex. Downloading.


  0%|          | 0/152.76 [00:01<?, ?it/s]


{'id': 'https://openalex.org/W4211011661',
 'doi': 'https://doi.org/10.1016/j.ocecoaman.2022.106078',
 'title': 'Assessing the validity of navigation risk assessments: A study of offshore wind farms in the UK',
 'display_name': 'Assessing the validity of navigation risk assessments: A study of offshore wind farms in the UK',
 'relevance_score': 454.002,
 'publication_year': 2022,
 'publication_date': '2022-02-09',
 'ids': {'openalex': 'https://openalex.org/W4211011661',
  'doi': 'https://doi.org/10.1016/j.ocecoaman.2022.106078'},
 'language': 'en',
 'primary_location': {'id': 'doi:10.1016/j.ocecoaman.2022.106078',
  'is_oa': True,
  'landing_page_url': 'https://doi.org/10.1016/j.ocecoaman.2022.106078',
  'pdf_url': None,
  'source': {'id': 'https://openalex.org/S8178725',
   'display_name': 'Ocean & Coastal Management',
   'issn_l': '0964-5691',
   'issn': ['0964-5691', '1873-524X'],
   'is_oa': False,
   'is_in_doaj': False,
   'is_core': True,
   'host_organization': 'https://openale

In [2]:
from slrrag.querier_openalex import query_openalex_and_standardize


query_openalex_and_standardize(input_query_openalex, n_paper_max=1)

Query : ((("offshore") AND ("infrastructure" OR "construction" OR "installation" OR "windfarm" OR "wind farm" OR "windmill" OR "wind turbine" OR "wind park" OR "wind energy" OR "platform" OR "quay" OR "wharf" OR "lidar")) OR ("bridge" OR "flotel" OR "Aquaculture farm" OR "Artificial Island" OR "Synthetic Island")) AND
(("risk" OR "safe" OR "safety") AND ("assess" OR "assessing" OR "assessment" OR "model" OR "modeling" OR "modelisation" OR "predict" OR "predicting" OR "prediction" OR "analyse" OR "analysing" OR "analyze" OR "analyzing" OR "study" OR "studies" OR "studying") AND ("ship" OR "vessel" OR "maritime traffic" OR "navigation" OR "navigational" OR "nautical"))



TypeError: 'NoneType' object is not iterable

query

In [ ]:
import pybliometrics
from pybliometrics.scopus import ScopusSearch

pybliometrics.init()  # Init for scopus
input_query_test_scopus = """
((("offshore") AND ("infrastructure" OR "construction" OR "installation" OR "windfarm" OR "wind farm" OR "windmill" OR "wind turbine" OR "wind park" OR "wind energy" OR "platform" OR "quay" OR "wharf" OR "lidar")) OR ("bridge" OR "flotel" OR "Aquaculture farm" OR "Artificial Island" OR "Synthetic Island")) AND
(("Navigation risk assessment" OR "Navigational risk assessment" OR "Navigation risk assessments" OR "Navigational risk assessments"))
"""
input_query_test_scopus = f"""
TITLE-ABS-KEY( {input_query_test_scopus})
"""
scopus_search = ScopusSearch(
    query=input_query_test_scopus, verbose=True, subscriber=True, sort="+relevancy"
)
scopus_search

In [ ]:
import pybliometrics
from pybliometrics.scopus import ScopusSearch

pybliometrics.init()  # Init for scopus
scopus_search = ScopusSearch(
    query='title("Navigational Risk assessment")',
    verbose=True,
    subscriber=False,
    sort="-relevance",
    refresh=True,
)
scopus_search

In [ ]:
data = [
    {
        "doi": paper.doi,
        "title": paper.title,
        "date": paper.coverDate,
        "abstract": paper.description,
        "id_scopus": paper.eid,
        "type": paper.subtypeDescription,
        "keywords_scopus": paper.authkeywords,
    }
    for paper in scopus_search.results
]
pd.DataFrame(data).assign(
    "relevace_oreder",
)

In [ ]:
[x.title for x in scopus_search.results]

In [ ]:
scopus_api_key = os.environ.get("SCOPUS_API_KEY")
assert scopus_api_key is not None
url = "https://api.elsevier.com/content/search/scopus"
params = {"query": 'title("navigational risk assessment")', "apiKey": scopus_api_key}
headers = {"Accept": "application/json"}

response = requests.get(url, headers=headers, params=params)

In [ ]:
dict_an = response.json()
dict_an

In [ ]:
dict_an["search-results"].keys()

In [ ]:
import pandas as pd

df_all = pd.DataFrame(dict_an["search-results"]["entry"])
df_all.iloc[0].T

In [ ]:
[x.authkeywords for x in scopus_search.results]

In [ ]:
fig = px.bar(
    l_test_openalex.groupby("topic_openalex")
    .size()
    .sort_values(ascending=False)
    .head(10)
    .rename("count")
    .reset_index(),
    y="topic_openalex",
    x="count",
)
# fig.update_xaxes(tickangle=90)
fig